In [15]:
# LOADING AND INSPECTING THE DATASET
import pandas as pd

file_path = r"C:\Users\user\OneDrive\Documents\Recommender System\DSA_4060_Week2_Movie_Ratings.csv"
df = pd.read_csv(file_path)

In [16]:
# The first 10 records
print("--- First 10 Records ---")
display(df.head(10))

--- First 10 Records ---


,user_id,movie_id,movie_title,rating,timestamp
0,U001,M001,Nairobi Nights,3,2026-01-01T08:00:00Z
1,U004,M001,Nairobi Nights,4,2026-01-06T09:00:00Z
2,U007,M001,Nairobi Nights,2,2026-01-11T10:00:00Z
3,U010,M001,Nairobi Nights,2,2026-01-16T11:00:00Z
4,U013,M001,Nairobi Nights,4,2026-01-21T12:00:00Z
5,U016,M001,Nairobi Nights,3,2026-01-26T13:00:00Z
6,U019,M001,Nairobi Nights,3,2026-01-31T14:00:00Z
7,U022,M001,Nairobi Nights,2,2026-02-05T15:00:00Z
8,U025,M001,Nairobi Nights,2,2026-02-10T16:00:00Z
9,U028,M001,Nairobi Nights,4,2026-02-15T17:00:00Z


In [17]:
# Dataset dimensions and data types
print("\n--- Dataset Dimensions ---")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n--- Data Types ---")
print(df.dtypes)


--- Dataset Dimensions ---
Rows: 509, Columns: 5

--- Data Types ---
user_id        object
movie_id       object
movie_title    object
rating          int64
timestamp      object
dtype: object


In [18]:
# Missing values and duplicate records
print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Duplicate Records Count ---")
print(df.duplicated().sum())


--- Missing Values ---
user_id        2
movie_id       1
movie_title    1
rating         0
timestamp      0
dtype: int64

--- Duplicate Records Count ---
3


In [ ]:
# CLEANING THE DATASET
# Removing duplicate records
df_clean = df.drop_duplicates()

# Removing records with missing user_id or movie_id
df_clean = df_clean.dropna(subset=['user_id', 'movie_id'])

# Confirming and cleaning ratings within the valid scale (1 to 5)
print("Unique ratings before cleaning:", sorted(df_clean['rating'].unique()))

# Filtering out ratings outside the 1-5 scale 
df_clean = df_clean[(df_clean['rating'] >= 1) & (df_clean['rating'] <= 5)]

print(f"Cleaned dataset shape: {df_clean.shape}")
print("Verified rating range:", df_clean['rating'].min(), "to", df_clean['rating'].max())

Unique ratings before cleaning: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]
Cleaned dataset shape: (500, 5)
Verified rating range: 1 to 5


In [ ]:
# EXPLORING THE INTERACTION DATA
# Core exploratory metrics
total_ratings = len(df_clean)
unique_users = df_clean['user_id'].nunique()
unique_movies = df_clean['movie_id'].nunique()

most_active_user = df_clean['user_id'].mode()[0]
most_rated_movie = df_clean['movie_title'].mode()[0]
average_rating = df_clean['rating'].mean()

print(f"Total number of ratings: {total_ratings}")
print(f"Number of unique users: {unique_users}")
print(f"Number of unique movies: {unique_movies}")
print(f"Most active user: {most_active_user}")
print(f"Most-rated movie: {most_rated_movie}")
print(f"Average rating in dataset: {average_rating:.2f}")

Total number of ratings: 500
Number of unique users: 50
Number of unique movies: 24
Most active user: U010
Most-rated movie: Mountain Echoes
Average rating in dataset: 3.45


In [11]:
# CREATING A USER-ITEM INTERACTION MATRIX
user_item_matrix = df_clean.pivot_table(index='user_id', columns='movie_id', values='rating')
print("--- User-Item Interaction Matrix (Sample) ---")
display(user_item_matrix.iloc[:5, :5])

--- User-Item Interaction Matrix (Sample) ---


movie_id,M001,M002,M003,M004,M005
user_id,,,,,
U001,3.0,2.0,NaN,NaN,4.0
U002,3.0,NaN,NaN,3.5,NaN
U003,NaN,NaN,NaN,NaN,4.0
U004,4.0,NaN,NaN,NaN,3.0
U005,2.0,4.0,NaN,3.0,NaN


In [12]:
# Calculating Sparsity
total_cells = unique_users * unique_movies
missing_cells = user_item_matrix.isna().sum().sum()
sparsity = missing_cells / total_cells

print(f"\nTotal Matrix Cells: {total_cells}")
print(f"Missing (NaN) Cells: {missing_cells}")
print(f"Sparsity: {sparsity:.4f} ({sparsity * 100:.2f}%)")


Total Matrix Cells: 1200
Missing (NaN) Cells: 770
Sparsity: 0.6417 (64.17%)


The sparsity calculation returned 64.17%, meaning that over 60% of the cells in the user-item interaction matrix are empty because most users have not rated the majority of the available movies. This high level of sparsity is typical in recommendation datasets, demonstrating that only a small fraction of the total possible user-movie interactions actually exist, which is a key challenge that collaborative filtering systems must handle.

In [13]:
# POPULARITY-BASED RECOMMENDATION SYSTEM

# Grouping by movie_id and movie_title to calculate metrics
movie_stats = df_clean.groupby(['movie_id', 'movie_title']).agg(
    average_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

In [14]:
# Filtering for movies with at least 10 ratings and sorting by average rating descending
top_10_recommendations = movie_stats[movie_stats['rating_count'] >= 10].sort_values(
    by='average_rating', ascending=False
).head(10)

print("--- Top 10 Popularity-Based Recommendations ---")
display(top_10_recommendations)

--- Top 10 Popularity-Based Recommendations ---


,movie_id,movie_title,average_rating,rating_count
20,M021,Market Day,4.466667,15
12,M013,Midnight in Mombasa,4.352941,17
13,M014,Code Breakers,4.250000,16
19,M020,Home Again,4.133333,15
18,M019,Future Africa,4.133333,15
6,M007,Hidden Lake,4.125000,16
5,M006,City of Tomorrow,3.911765,34
4,M005,Mountain Echoes,3.864865,37
11,M012,The Final Goal,3.764706,17
3,M004,Coastal Dreams,3.666667,27


In [10]:
top_10_recommendations.to_csv('top_10_recommendations.csv', index=False)
print("\nSuccessfully saved top 10 recommendations to 'top_10_recommendations.csv'!")


Successfully saved top 10 recommendations to 'top_10_recommendations.csv'!


The popularity-based recommendations appear at the top because they strike a  balance between high user satisfaction and statistical reliability, ranking movies with the highest average scores only after filtering out titles with fewer than 10 ratings to prevent skewed results from small sample sizes. 

A primary advantage of this approach is its ability to handle the cold-start problem seamlessly, allowing new users or platforms with no prior interaction history to immediately receive sensible, trending suggestions. 

Limitations: 

-> It provides entirely generic recommendations that ignore individual user preferences or niche tastes 

-> Creates a self-reinforcing feedback loop where already popular items receive continuous exposure while unique or less-mainstream titles remain undiscovered.